# Rainfall and vegetation response in North Turkana

Use 2015–2022 CHIRPS rainfall and quality-screened MODIS NDVI to compare correlations at lags of 0–4 months, before and after removing the seasonal cycle. Read the student guide for site selection and interpretation. Run each cell in order.

## 1 Connect to Earth Engine
Use your own credentials and an Earth Engine enabled Google Cloud project. Enter the project ID below.

In [ ]:
import ee
import pandas as pd
import matplotlib.pyplot as plt

project_id = input('Earth Engine Cloud project ID: ').strip()
ee.Authenticate()
ee.Initialize(project=project_id)
print(ee.String('Connected to Earth Engine').getInfo())

## 2 Set your study point and load the products
Enter your documented rangeland coordinates in decimal degrees, longitude first. Rainfall represents the CHIRPS grid cell and NDVI represents the 2.5 km buffer.

In [ ]:
longitude = float(input('Longitude: '))
latitude = float(input('Latitude: '))
point = ee.Geometry.Point([longitude, latitude])
buffer = point.buffer(2500)

chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').select('precipitation')
modis = ee.ImageCollection('MODIS/061/MOD13Q1')
rain_grid = chirps.first().projection()
ndvi_grid = modis.first().select('NDVI').projection()

def screen_ndvi(image):
    good = image.select('SummaryQA').eq(0)
    land = image.select('DetailedQA').rightShift(11).bitwiseAnd(7).eq(1)
    return (image.select('NDVI').multiply(0.0001)
            .updateMask(good.And(land))
            .copyProperties(image, ['system:time_start']))

ndvi = modis.map(screen_ndvi)

## 3 Inspect the sampling area
Change the displayed month and rerun this cell to compare maps. The outline marks the area used for NDVI averaging.

In [ ]:
import folium

map_month = ee.Date('2020-04-01')
image = ndvi.filterDate(map_month, map_month.advance(1, 'month')).mean().clip(buffer.buffer(10000))
tiles = image.getMapId({'min': 0, 'max': 0.5, 'palette': ['f5e8c7', '78b56c', '155d27']})
site_map = folium.Map(location=[latitude, longitude], zoom_start=12)
folium.TileLayer(tiles=tiles['tile_fetcher'].url_format,
                 attr='NASA MODIS via Google Earth Engine', name='Monthly NDVI').add_to(site_map)
folium.Circle([latitude, longitude], radius=2500, color='yellow', fill=False).add_to(site_map)
folium.Marker([latitude, longitude], tooltip='Rainfall sampling point').add_to(site_map)
folium.LayerControl().add_to(site_map)
display(site_map)

## 4 Build the monthly table
Sum daily rainfall to mm per month. Average valid NDVI composites by timestamp month, then average their pixels in the buffer. Inspect missing values and preserve their calendar rows.

In [ ]:
start = ee.Date('2015-01-01')

def monthly_row(offset):
    begin = start.advance(offset, 'month')
    end = begin.advance(1, 'month')
    rain = chirps.filterDate(begin, end).sum().rename('rain_mm')
    green = ndvi.filterDate(begin, end).mean().rename('ndvi')
    rainfall = rain.reduceRegion(ee.Reducer.first(), point, crs=rain_grid)
    greenness = green.reduceRegion(ee.Reducer.mean(), buffer, crs=ndvi_grid)
    return ee.Feature(None, rainfall.combine(greenness)).set('month', begin.format('YYYY-MM'))

table = ee.FeatureCollection(ee.List.sequence(0, 95).map(monthly_row)).sort('month')
records = [feature['properties'] for feature in table.getInfo()['features']]
data = pd.DataFrame(records).reindex(columns=['month', 'rain_mm', 'ndvi'])
data = data.sort_values('month').reset_index(drop=True)
expected = pd.period_range('2015-01', '2022-12', freq='M').astype(str).tolist()
assert data['month'].tolist() == expected, 'Check the monthly calendar.'
print(data.isna().sum())
display(data.head())
data.set_index(pd.to_datetime(data['month']))[['rain_mm', 'ndvi']].plot(subplots=True, figsize=(10, 5))
plt.tight_layout()
plt.show()
data.to_csv('turkana_rain_ndvi_monthly.csv', index=False)

## 5 Compare raw and seasonal anomaly correlations
A positive lag pairs earlier rain with later NDVI. For example, January rainfall pairs with February NDVI at lag 1. The anomaly subtracts the average for that calendar month. Compare peak shape and valid pair counts.

In [ ]:
calendar_month = pd.to_datetime(data['month']).dt.month
series = data[['rain_mm', 'ndvi']]
anomalies = series - series.groupby(calendar_month).transform('mean')

def correlation_at_lag(values, lag):
    paired = pd.concat([values['rain_mm'], values['ndvi'].shift(-lag)], axis=1).dropna()
    correlation = paired['rain_mm'].corr(paired['ndvi']) if len(paired) >= 2 else float('nan')
    return correlation, len(paired)

results = pd.DataFrame([
    (lag, *correlation_at_lag(series, lag), *correlation_at_lag(anomalies, lag))
    for lag in range(5)
], columns=['lag_months', 'raw_r', 'raw_pairs', 'anomaly_r', 'anomaly_pairs'])
display(results)
results.plot(x='lag_months', y=['raw_r', 'anomaly_r'], marker='o', xticks=range(5))
plt.ylabel('Pearson correlation')
plt.show()
results.to_csv('turkana_lag_results.csv', index=False)

## 6 Save your work
Download the CSV files from Colab's Files panel. Verify one lag manually using the student workbook. Report whether a clear positive peak persists after removing the seasonal cycle and include the valid pair counts. Save your own notebook copy before making changes.